# Experimento 4 V3 — Chain of Thought (Modelos Pequenos)

**Modelos:** Mistral 7B, Qwen 2.5 7B, Gemma 2 9B  
**Split:** 80/20 | **Folds:** 5 com media e desvio padrao  
**Checkpoints:** salvos a cada 50 redacoes por fold. Retomada automatica ao reexecutar.  
**Obs:** os feedbacks gerados sao salvos nos CSVs e serao usados pelo Exp5.

In [ ]:
# Celula 1 - Instalacao
!pip install -q transformers accelerate bitsandbytes gdown scikit-learn matplotlib peft optuna
print('Instalado!')

In [ ]:
# Celula 2 - Imports e autenticacao
import re, gc, json, time, random, os
import numpy as np
import pandas as pd
import torch
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, cohen_kappa_score, f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata
from huggingface_hub import login

optuna.logging.set_verbosity(optuna.logging.WARNING)

HF_TOKEN     = userdata.get('HF_TOKEN')
N_FOLDS      = 5
RANDOM_STATE = 42

login(token=HF_TOKEN)
print('Autenticado!')

In [ ]:
# Celula 3 - Dataset e K-Fold (80/20)
import gdown
import pandas as pd
import re
import ast
from sklearn.model_selection import StratifiedKFold

novo_id = '1chJZo8L4s3Zuv1nzHrZa3b6ycf0ePQhv'
gdown.download(
    f'https://drive.google.com/uc?id={novo_id}',
    'meu_dataset.csv', quiet=False
)
df_enem = pd.read_csv('meu_dataset.csv')

df_enem[['c1', 'c2', 'c3', 'c4', 'c5']] = df_enem['competence'].apply(ast.literal_eval).tolist()


def limpar(texto):
    if pd.isna(texto): return ''
    texto = str(texto).strip("[]'\" ")
    texto = texto.replace('\n', ' ')
    texto = re.sub(r'\[[A-Z/]+\]', '', texto)
    texto = re.sub(r'\{[a-z]+\}', '', texto)
    return re.sub(r'\s+', ' ', texto).strip()


df_enem['essay_limpo'] = df_enem['essay'].apply(limpar)
df_enem = df_enem[df_enem['score'] > 0].reset_index(drop=True)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
df_enem['fold'] = -1
for i, (tr, te) in enumerate(skf.split(df_enem, df_enem['score'])):
    df_enem.loc[te, 'fold'] = i

print(f'Total: {len(df_enem)} | Folds: {N_FOLDS} | Teste por fold: ~{len(df_enem) // N_FOLDS}')

In [ ]:
# Celula 4 - Funcoes auxiliares

def extrair_notas(resposta):
    try:
        texto = re.sub(r'```json|```', '', str(resposta))
        i = texto.find('{')
        j = texto.rfind('}') + 1
        if i == -1 or j <= i: return None
        dados = json.loads(texto[i:j])
        comps = ['C1', 'C2', 'C3', 'C4', 'C5']
        if all(c in dados for c in comps):
            if max(dados[c] for c in comps) <= 20:
                for c in comps: dados[c] *= 10
            dados['Nota_Total'] = sum(dados[c] for c in comps)
            return dados
        if 'Nota_Total' in dados: return dados
    except: pass
    return None


def extrair_notas_markdown(resposta):
    try:
        texto = str(resposta)
        comps = {}
        for c in ['C1', 'C2', 'C3', 'C4', 'C5']:
            m = re.search(rf'{c}[^:]*:\s*(\d+)', texto)
            if m: comps[c] = int(m.group(1))
        if len(comps) == 5:
            if max(comps.values()) <= 20:
                for c in comps: comps[c] *= 10
            comps['Nota_Total'] = sum(comps.values())
            return comps
        m = re.search(r'(?:Total|Nota\s*Total|Nota\s*Final)[^\d]*(\d{3,4})', texto, re.I)
        if m: return {'Nota_Total': int(m.group(1))}
    except: pass
    return None


def extrair(resposta):
    return extrair_notas(resposta) or extrair_notas_markdown(resposta)


def extrair_feedback(resposta):
    try:
        t = str(resposta)
        j = t.rfind('}')
        fb = t[j + 1:].strip() if j >= 0 else ''
        if len(fb) < 20:
            inicio = t.find('{')
            fb = t[:inicio].strip() if inicio >= 0 else ''
        return fb if len(fb) > 20 else 'Feedback nao disponivel'
    except:
        return 'Feedback nao disponivel'


def calcular_metricas(y_true, y_pred, nome):
    yt = np.array(y_true, dtype=float)
    yp = np.array(y_pred, dtype=float)
    mae  = mean_absolute_error(yt, yp)
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    def disc(n): return np.clip(np.round(np.array(n) / 40).astype(int), 0, 25)
    try:    qwk = cohen_kappa_score(disc(yt), disc(yp), weights='quadratic')
    except: qwk = float('nan')
    try:    f1  = f1_score(disc(yt), disc(yp), average='weighted', zero_division=0)
    except: f1  = float('nan')
    print(f'\n{"="*55}')
    print(f'METRICAS — {nome}')
    print(f'{"="*55}')
    print(f'  Amostras : {len(yt)}')
    print(f'  MAE      : {mae:.4f}')
    print(f'  RMSE     : {rmse:.4f}')
    print(f'  QWK      : {qwk:.4f}')
    print(f'  F1 Score : {f1:.4f}')
    print(f'{"="*55}')
    return {'modelo': nome, 'mae': mae, 'rmse': rmse, 'qwk': qwk, 'f1': f1, 'n': len(yt)}


def agregar_folds(resultados_folds, nome):
    validos = [r for r in resultados_folds if r is not None]
    if not validos: return None
    metricas = ['mae', 'rmse', 'qwk', 'f1']
    agregado = {'modelo': nome}
    for m in metricas:
        vals = [r[m] for r in validos if not np.isnan(r[m])]
        agregado[m]          = float(np.mean(vals)) if vals else float('nan')
        agregado[m + '_std'] = float(np.std(vals))  if vals else float('nan')
    agregado['n_folds'] = len(validos)
    print(f'\n{"="*60}')
    print(f'MEDIA {N_FOLDS} FOLDS — {nome}')
    print(f'{"="*60}')
    for m in metricas:
        print(f'  {m.upper():<6}: {agregado[m]:.4f} +/- {agregado[m+"_std"]:.4f}')
    print(f'  Folds validos: {agregado["n_folds"]}/{N_FOLDS}')
    print(f'{"="*60}')
    return agregado


print('Funcoes auxiliares carregadas!')

In [ ]:
# Celula 5 - Inferencia e carregamento

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)


def carregar_modelo(nome_modelo):
    print(f'Carregando {nome_modelo}...')
    tok = AutoTokenizer.from_pretrained(nome_modelo, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(
        nome_modelo, quantization_config=bnb_config,
        device_map='auto', trust_remote_code=True
    )
    m.eval()
    print('Carregado!')
    return tok, m


def liberar(m, tok):
    del m, tok
    gc.collect()
    torch.cuda.empty_cache()
    print('Memoria GPU liberada.')


def gerar_prompt_cot(redacao, max_chars=None):
    if max_chars:
        redacao = redacao[:max_chars]
    return (
        'Voce e um avaliador oficial de redacoes do ENEM.\n'
        'Avalie a redacao seguindo a escala: 0, 40, 80, 120, 160 ou 200 pontos por competencia.\n\n'
        'COMPETENCIAS:\n'
        '- C1 (Norma Culta): dominio da norma padrao (0-200)\n'
        '- C2 (Tema/Estrutura): adequacao ao tema e estrutura (0-200)\n'
        '- C3 (Argumentacao): selecao e organizacao de argumentos (0-200)\n'
        '- C4 (Coesao): uso de mecanismos linguisticos (0-200)\n'
        '- C5 (Proposta de Intervencao): proposta com agente, acao, meio, efeito (0-200)\n\n'
        'REDACAO:\n' + redacao + '\n\n'
        'Siga EXATAMENTE estas etapas:\n\n'
        'PASSO 1 - ANALISE:\nAnalise cada competencia separadamente.\n\n'
        'PASSO 2 - NOTAS:\nAtribua as notas no formato JSON:\n'
        '{"C1": valor, "C2": valor, "C3": valor, "C4": valor, "C5": valor, "Nota_Total": soma}\n\n'
        'PASSO 3 - FEEDBACK:\nEscreva 2-3 paragrafos de feedback construtivo.'
    )


def inf_local_cot(tok, m, prompt, temp=0.2, max_tok=700):
    msgs = [{'role': 'user', 'content': prompt}]
    try:
        txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except:
        txt = prompt
    inp = tok(txt, return_tensors='pt', truncation=True, max_length=3072).to(m.device)
    with torch.no_grad():
        out = m.generate(
            **inp, max_new_tokens=max_tok,
            temperature=temp, do_sample=True,
            pad_token_id=tok.pad_token_id
        )
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()


print('Funcoes de inferencia carregadas!')

In [ ]:
# Celula 6 - Optuna e runner CoT por fold

def ckpt_path(nome_curto, fold):
    return f'ck_exp4_{nome_curto}_fold{fold}.csv'


def fold_completo(nome_curto, fold):
    path = ckpt_path(nome_curto, fold)
    if not os.path.exists(path): return False
    df_ck = pd.read_csv(path)
    n_te  = len(df_enem[df_enem['fold'] == fold])
    return len(df_ck.dropna(subset=['pred_total'])) >= n_te * 0.95


def obj_optuna_cot(trial, fn_inf):
    temp      = trial.suggest_float('temp', 0.05, 0.5)
    max_chars = trial.suggest_int('max_chars', 500, 2000, step=250)
    df_val    = df_enem[df_enem['fold'] == 1].head(10)
    yp, yt    = [], []
    for _, row in df_val.iterrows():
        try:
            prompt = gerar_prompt_cot(row['essay_limpo'][:max_chars])
            resp   = fn_inf(prompt, temp)
            notas  = extrair(resp)
            if notas and 'Nota_Total' in notas:
                yp.append(notas['Nota_Total'])
                yt.append(row['score'])
        except: pass
    if len(yp) < 5: raise optuna.TrialPruned()
    def disc(n): return np.clip(np.round(np.array(n) / 40).astype(int), 0, 25)
    try:    return cohen_kappa_score(disc(yt), disc(yp), weights='quadratic')
    except: return -1.0


def rodar_cot_fold(nome, nome_curto, fn_inf, params, fold, df_te):
    temp      = params.get('temp', 0.2)
    max_chars = params.get('max_chars', None)
    path      = ckpt_path(nome_curto, fold)

    if os.path.exists(path):
        df_ck     = pd.read_csv(path)
        ja_feitos = set(df_ck['index_redacao'].tolist())
        print(f'  Checkpoint fold {fold}: {len(ja_feitos)} ja processadas')
    else:
        df_ck     = pd.DataFrame(columns=['index_redacao', 'score', 'pred_total', 'feedback'])
        ja_feitos = set()

    pendentes = df_te[~df_te.index.isin(ja_feitos)]
    print(f'  Pendentes fold {fold}: {len(pendentes)}/{len(df_te)}')

    novos = []
    for idx, (i, row) in enumerate(pendentes.iterrows()):
        try:
            redacao = row['essay_limpo'][:max_chars] if max_chars else row['essay_limpo']
            prompt  = gerar_prompt_cot(redacao)
            resp    = fn_inf(prompt, temp)
            notas   = extrair(resp)
            pred    = notas['Nota_Total'] if notas else None
            fb      = extrair_feedback(resp)
            novos.append({'index_redacao': i, 'score': row['score'], 'pred_total': pred, 'feedback': fb})
        except:
            novos.append({'index_redacao': i, 'score': row['score'], 'pred_total': None, 'feedback': None})

        if (idx + 1) % 50 == 0:
            df_ck = pd.concat([df_ck, pd.DataFrame(novos)], ignore_index=True)
            df_ck.to_csv(path, index=False)
            novos = []
            print(f'    Checkpoint: {idx + 1 + len(ja_feitos)}/{len(df_te)}')

    if novos:
        df_ck = pd.concat([df_ck, pd.DataFrame(novos)], ignore_index=True)
        df_ck.to_csv(path, index=False)

    df_v = df_ck.dropna(subset=['pred_total'])
    print(f'  Validas fold {fold}: {len(df_v)}/{len(df_te)}')
    if len(df_v) > 0:
        return calcular_metricas(df_v['score'].tolist(), df_v['pred_total'].tolist(), f'{nome} CoT fold{fold}')
    return None


print('Runners carregados!')

In [ ]:
# Celula 7 - Mistral 7B
nome_modelo  = 'mistralai/Mistral-7B-Instruct-v0.3'
nome_curto   = 'mistral'
nome_display = 'Mistral 7B'

print(f'\n{"="*60}')
print(f'MISTRAL 7B — EXP4 Chain of Thought')
print(f'{"="*60}')

tok, m = carregar_modelo(nome_modelo)
fn_cot = lambda p, t: inf_local_cot(tok, m, p, temp=t)

study_mistral = optuna.create_study(
    study_name='optuna_mistral_cot',
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5),
    storage='sqlite:///optuna_mistral_cot.db',
    load_if_exists=True
)
)
if len(study_mistral.trials) < 8:
    study_mistral.optimize(lambda t: obj_optuna_cot(t, fn_cot), n_trials=8, catch=(Exception,))
params_mistral = study_mistral.best_params
print(f'Melhores params: {params_mistral}')

resultados_mistral = []
for fold in range(N_FOLDS):
    df_te = df_enem[df_enem['fold'] == fold].reset_index(drop=True)
    if fold_completo(nome_curto, fold):
        print(f'Fold {fold} ja completo, carregando CSV...')
        df_ck = pd.read_csv(ckpt_path(nome_curto, fold)).dropna(subset=['pred_total'])
        res   = calcular_metricas(df_ck['score'].tolist(), df_ck['pred_total'].tolist(), f'{nome_display} CoT fold{fold}')
    else:
        res = rodar_cot_fold(nome_display, nome_curto, fn_cot, params_mistral, fold, df_te)
    resultados_mistral.append(res)

liberar(m, tok)
media_mistral = agregar_folds(resultados_mistral, f'{nome_display} (CoT)')
pd.DataFrame([r for r in resultados_mistral if r]).to_csv(f'resultados_exp4_{nome_curto}_folds.csv', index=False)
print(f'\nMistral 7B Exp4 completo!')

In [ ]:
# Celula 8 - Qwen 2.5 7B
nome_modelo  = 'Qwen/Qwen2.5-7B-Instruct'
nome_curto   = 'qwen7b'
nome_display = 'Qwen 2.5 7B'

print(f'\n{"="*60}')
print(f'QWEN 2.5 7B — EXP4 Chain of Thought')
print(f'{"="*60}')

tok, m = carregar_modelo(nome_modelo)
fn_cot = lambda p, t: inf_local_cot(tok, m, p, temp=t)

study_qwen7b = optuna.create_study(
    study_name='optuna_qwen7b_cot',
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5),
    storage='sqlite:///optuna_qwen7b_cot.db',
    load_if_exists=True
)
)
if len(study_qwen7b.trials) < 8:
    study_qwen7b.optimize(lambda t: obj_optuna_cot(t, fn_cot), n_trials=8, catch=(Exception,))
params_qwen7b = study_qwen7b.best_params
print(f'Melhores params: {params_qwen7b}')

resultados_qwen7b = []
for fold in range(N_FOLDS):
    df_te = df_enem[df_enem['fold'] == fold].reset_index(drop=True)
    if fold_completo(nome_curto, fold):
        print(f'Fold {fold} ja completo, carregando CSV...')
        df_ck = pd.read_csv(ckpt_path(nome_curto, fold)).dropna(subset=['pred_total'])
        res   = calcular_metricas(df_ck['score'].tolist(), df_ck['pred_total'].tolist(), f'{nome_display} CoT fold{fold}')
    else:
        res = rodar_cot_fold(nome_display, nome_curto, fn_cot, params_qwen7b, fold, df_te)
    resultados_qwen7b.append(res)

liberar(m, tok)
media_qwen7b = agregar_folds(resultados_qwen7b, f'{nome_display} (CoT)')
pd.DataFrame([r for r in resultados_qwen7b if r]).to_csv(f'resultados_exp4_{nome_curto}_folds.csv', index=False)
print(f'\nQwen 2.5 7B Exp4 completo!')

In [ ]:
# Celula 9 - Gemma 2 9B
nome_modelo  = 'google/gemma-2-9b-it'
nome_curto   = 'gemma'
nome_display = 'Gemma 2 9B'

print(f'\n{"="*60}')
print(f'GEMMA 2 9B — EXP4 Chain of Thought')
print(f'{"="*60}')

tok, m = carregar_modelo(nome_modelo)
fn_cot = lambda p, t: inf_local_cot(tok, m, p, temp=t)

study_gemma = optuna.create_study(
    study_name='optuna_gemma_cot',
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5),
    storage='sqlite:///optuna_gemma_cot.db',
    load_if_exists=True
)
)
if len(study_gemma.trials) < 8:
    study_gemma.optimize(lambda t: obj_optuna_cot(t, fn_cot), n_trials=8, catch=(Exception,))
params_gemma = study_gemma.best_params
print(f'Melhores params: {params_gemma}')

resultados_gemma = []
for fold in range(N_FOLDS):
    df_te = df_enem[df_enem['fold'] == fold].reset_index(drop=True)
    if fold_completo(nome_curto, fold):
        print(f'Fold {fold} ja completo, carregando CSV...')
        df_ck = pd.read_csv(ckpt_path(nome_curto, fold)).dropna(subset=['pred_total'])
        res   = calcular_metricas(df_ck['score'].tolist(), df_ck['pred_total'].tolist(), f'{nome_display} CoT fold{fold}')
    else:
        res = rodar_cot_fold(nome_display, nome_curto, fn_cot, params_gemma, fold, df_te)
    resultados_gemma.append(res)

liberar(m, tok)
media_gemma = agregar_folds(resultados_gemma, f'{nome_display} (CoT)')
pd.DataFrame([r for r in resultados_gemma if r]).to_csv(f'resultados_exp4_{nome_curto}_folds.csv', index=False)
print(f'\nGemma 2 9B Exp4 completo!')

In [ ]:
# Celula 10 - Consolidacao final
import matplotlib.pyplot as plt

todos = [r for r in [media_mistral, media_qwen7b, media_gemma] if r is not None]
df_res = pd.DataFrame(todos)

print('\n' + '='*85)
print(f'{"Modelo":<35} {"Folds":>6} {"MAE":>9} {"RMSE":>9} {"QWK":>9} {"F1":>9}')
print('-'*85)
for _, row in df_res.iterrows():
    print(
        f'{row["modelo"]:<35} '
        f'{int(row["n_folds"]):>6} '
        f'{row["mae"]:>7.3f}+/-{row["mae_std"]:.3f} '
        f'{row["rmse"]:>7.3f}+/-{row["rmse_std"]:.3f} '
        f'{row["qwk"]:>7.3f}+/-{row["qwk_std"]:.3f} '
        f'{row["f1"]:>7.3f}+/-{row["f1_std"]:.3f}'
    )
print('='*85)

df_res.to_csv('resultados_exp4_v3_final.csv', index=False)
print('\nCSV salvo: resultados_exp4_v3_final.csv')

cores = ['#4C72B0', '#DD8452', '#55A868']
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(
    f'Experimento 4 V3 — Chain of Thought\nMedia de {N_FOLDS} Folds (80/20)',
    fontsize=14, fontweight='bold'
)
for ax, (titulo, coluna) in zip(axes.flatten(), [
    ('MAE (menor = melhor)', 'mae'),
    ('RMSE (menor = melhor)', 'rmse'),
    ('QWK (maior = melhor)', 'qwk'),
    ('F1 Score (maior = melhor)', 'f1')
]):
    barras = ax.bar(df_res['modelo'], df_res[coluna], color=cores, edgecolor='white')
    ax.errorbar(
        range(len(df_res)), df_res[coluna],
        yerr=df_res[coluna + '_std'],
        fmt='none', color='black', capsize=5, linewidth=1.5
    )
    for b in barras:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.005,
                f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_ylabel('Valor')
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('grafico_exp4_v3_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo: grafico_exp4_v3_final.png')